# Thyroid Disease Prediction

This notebook implements a complete solution for predicting thyroid disease based on patient medical parameters. It includes data preprocessing, feature analysis, model training, and a prediction function.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Load the dataset
df = pd.read_csv('cleaned_dataset_Thyroid1.csv')
print(df.head())
print(df.info())

In [ ]:
# Handle missing values
# Assuming numerical columns for imputation
numerical_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Impute numerical with mean
imputer_num = SimpleImputer(strategy='mean')
df[numerical_cols] = imputer_num.fit_transform(df[numerical_cols])

# Impute categorical with most frequent
imputer_cat = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = imputer_cat.fit_transform(df[categorical_cols])

print(df.isnull().sum())

In [ ]:
# Encode categorical variables
# Assuming target is 'binaryClass' or similar, map to 'Thyroid Positive' and 'Thyroid Negative'
# Adjust based on actual column names
if 'binaryClass' in df.columns:
    df['binaryClass'] = df['binaryClass'].map({'P': 'Thyroid Positive', 'N': 'Thyroid Negative'})

# Separate features and target
X = df.drop('binaryClass', axis=1)
y = df['binaryClass']

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# One-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ])

X_processed = preprocessor.fit_transform(X)

In [ ]:
# Feature analysis
# Correlation heatmap for numerical features
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Distribution of target
sns.countplot(x=y)
plt.title('Target Distribution')
plt.show()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Models
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'KNN': KNeighborsClassifier(),
    'Gradient Boosting': GradientBoostingClassifier()
}

# Train and evaluate
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'{name} Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred))

# Select best model (e.g., Random Forest)
best_model = RandomForestClassifier()
best_model.fit(X_train, y_train)

In [ ]:
# Add this cell after the model training cell

import pickle

# Save the best model
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save the preprocessor
with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

# Save the label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

In [ ]:
import os
files = ['best_model.pkl', 'preprocessor.pkl', 'label_encoder.pkl']
for file in files:
    if os.path.exists(file):
        print(f"✓ {file} exists")
    else:
        print(f"✗ {file} NOT found")

In [ ]:
# Prediction function
def predict_thyroid(age, sex, tsh, t3, tt4, t4u, fti, tbg=None):
    # Create input DataFrame (adjust columns based on dataset)
    input_data = pd.DataFrame({
        'age': [age],
        'sex': [sex],
        'TSH': [tsh],
        'T3': [t3],
        'TT4': [tt4],
        'T4U': [t4u],
        'FTI': [fti],
        'TBG': [tbg] if tbg else [np.nan]
    })
    
    # Preprocess
    input_processed = preprocessor.transform(input_data)
    
    # Predict
    pred = best_model.predict(input_processed)
    result = le.inverse_transform(pred)[0]
    return result

# Example usage
# print(predict_thyroid(30, 'F', 1.5, 2.0, 100, 1.0, 100))